# Posterior Check via MIRA

Evaluate the **conditional (posterior) accuracy** of the strong-lensing source
reconstructions with [MIRA](../mira-score) (*A Score for Conditional Distribution
Accuracy and Model Comparison*).

For each true source galaxy we have a set of posterior draws produced by
`src/sample.py` (`samples/posterior_draws.pt`). MIRA asks whether, across **many**
such (truth, posterior) pairs, the truth behaves like a typical posterior draw
(good calibration) rather than sitting outside a too-tight posterior
(over-confident) or being swamped by a too-wide one (under-confident). It also
lets us **compare models** (e.g. different checkpoints or noise levels) on the
same set of sources.

**MIRA API** (`mira_score.mira`):

```
mira(truth, posterior, num_runs=100, norm=False, ...) -> (score (M,), std (M,))
    truth      : (T, q)         T ground-truth vectors in q dims
    posterior  : (M, T, S, q)   M models, T truths, S samples each, q dims
```

so we map: **one truth = one source**, **q = image features** (PCA of the
flattened image), **S = posterior draws per source**, **M = models**.

### Testing without a checkpoint / HPC data
- **Section 3** is a self-contained **MIRA smoke test** (Gaussian toy) that runs
  immediately and confirms the score direction.
- **Section 4** optionally **fabricates `posterior_draws.pt` files** so the *real*
  loading + PCA + scoring path (Sections 5-8) also runs with no model. Set
  `MAKE_TOY_DATA = True` and Run All.
- With real runs present under `outputs/`, leave `MAKE_TOY_DATA = False` and the
  same cells score your actual posteriors.

## 1. Setup

In [ ]:
import sys, glob
from pathlib import Path

# Locate the project repo (dir with src/sample.py) and the MIRA repo, so the
# notebook runs regardless of the kernel's working directory. Wilkes3 absolute
# paths are added as fallbacks (same convention as PQMassPriorCheck.ipynb).
_repo_candidates = [Path.cwd(), *Path.cwd().parents,
                    Path("/rds/user/yd388/hpc-work/DIS-Project-Lensed-Galaxy")]
REPO = next(p for p in _repo_candidates if (p / "src" / "sample.py").exists())
sys.path.insert(0, str(REPO / "src"))

_mira_candidates = [REPO.parent / "mira-score",
                    Path("/rds/user/yd388/hpc-work/mira-score"),
                    Path.home() / "mira-score"]
MIRA_REPO = next((p for p in _mira_candidates
                  if (p / "src" / "mira_score" / "mira.py").exists()), None)

try:
    from mira_score import mira, mira_bootstrap, get_device
except ModuleNotFoundError:
    if MIRA_REPO is None:
        raise ModuleNotFoundError(
            "mira_score not importable and the mira-score repo was not found next "
            "to the project. Either `pip install -e ../mira-score` or set MIRA_REPO.")
    sys.path.insert(0, str(MIRA_REPO / "src"))
    from mira_score import mira, mira_bootstrap, get_device

import numpy as np
import torch
import matplotlib.pyplot as plt

device = get_device()
print(f"repo:      {REPO}")
print(f"mira repo: {MIRA_REPO}")
print(f"device:    {device}")

## 2. Configuration

In [ ]:
# MIRA compares one or more MODELS, each evaluated on the SAME set of truths.
# Here one "truth" = one true source galaxy; its posterior draws are the samples
# src/sample.py wrote for that source (samples/posterior_draws.pt).
#
# For a meaningful calibration test you want MANY truths -- i.e. many source
# picks (run src/sample.py with several --pick / --output_dir). MIRA on a single
# truth is not informative.
#
# MODELS maps model name -> list of run dirs (each an --output_dir containing
# samples/posterior_draws.pt). By default we AUTO-DISCOVER every such run under
# outputs/ and treat them as one model. To compare models, set e.g.
#   MODELS = {"sigma_0.1": [...run dirs...], "sigma_0.3": [...run dirs...]}
# where every model was sampled on the SAME sources (same picks).

def discover_runs():
    found = sorted(glob.glob(str(REPO / "outputs" / "**" / "samples" /
                                 "posterior_draws.pt"), recursive=True))
    return [str(Path(p).parent.parent) for p in found]

_run_dirs = discover_runs()
MODELS = {"model": _run_dirs}      # <- replace to compare several models

USE_PCA       = True   # project flattened images to PCA_DIMS before scoring
                       # (recommended; raw images are too high-dim for
                       # distance-based scores -- see the PQMass notebook).
PCA_DIMS      = 50
NORM          = True   # MIRA min-max normalisation of the score space
NUM_RUNS      = 200    # Monte-Carlo replications inside mira()
NUM_BOOTSTRAP = 200    # resamples for mira_bootstrap() uncertainty
ADD_BASELINES = True   # add over-/under-confident posteriors as a calibration ruler
SEED          = 21

torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Discovered {len(_run_dirs)} posterior run(s) under outputs/:")
for r in _run_dirs:
    print("  ", r)
HAVE_DATA = len(_run_dirs) > 0
if not HAVE_DATA:
    print("\nNo posterior_draws.pt found yet. Run the toy generator in Section 4 "
          "(MAKE_TOY_DATA=True) to exercise the full pipeline, or point MODELS at "
          "real runs. The Section 3 smoke test runs regardless.")

## 3. Synthetic smoke test (runs now, no files)

Verify that MIRA imports and our calling convention is right, and confirm the
**score direction**: with truths drawn as `theta ~ N(m, I)` and posteriors as
`N(m, spread^2 I)`, the **calibrated** case (`spread == 1`) should get the
highest MIRA score, above the over-confident (too tight) and under-confident
(too wide) posteriors.

In [ ]:
def _make_gaussian_case(spread, T=200, S=100, q=5, seed=0):
    g = torch.Generator().manual_seed(seed)
    centers = torch.randn(T, q, generator=g)                # per-truth mean m_t
    truth   = centers + torch.randn(T, q, generator=g)      # truth ~ N(m_t, I)
    samples = (centers[:, None, :] +
               spread * torch.randn(T, S, q, generator=g))  # post ~ N(m_t, spread^2 I)
    return truth, samples

truth_s, cal   = _make_gaussian_case(1.0, seed=1)   # calibrated
_,       over  = _make_gaussian_case(0.3, seed=1)   # over-confident (too tight)
_,       under = _make_gaussian_case(3.0, seed=1)   # under-confident (too wide)
post_s  = torch.stack([cal, over, under], dim=0)    # (M=3, T, S, q)
names_s = ["calibrated", "over-confident (0.3x)", "under-confident (3x)"]

score_s, std_s = mira(truth_s, post_s, num_runs=NUM_RUNS, norm=False, device=device)
for n, m, sd in zip(names_s, score_s.tolist(), std_s.tolist()):
    print(f"  {n:24s}  MIRA = {m:.4f} +/- {sd:.4f}")
print(f"\nHighest MIRA: {names_s[int(torch.argmax(score_s))]!r}  "
      f"(expected 'calibrated' -> higher score = better calibrated)")

## 4. Optional: generate a toy dataset

Fabricate `posterior_draws.pt` files that match the schema written by
`src/sample.py`, so Sections 5-8 run end-to-end **without a checkpoint or HPC
samples**. Each toy source is a smooth Gaussian "galaxy" blob; the truth and the
`S` posterior draws are independent draws of `N(m_t, TAU^2)` -- exchangeable, and
therefore calibrated -- so the toy posterior should score at or above the
over-/under-confident baselines in Section 7. `outputs/` is gitignored, so this
is throwaway scratch.

In [ ]:
MAKE_TOY_DATA = False        # <- set True, then Run All, to test without a checkpoint

if MAKE_TOY_DATA:
    import shutil
    T_TOY, S_TOY, H_TOY, TAU = 16, 64, 32, 0.15
    toy_root = REPO / "outputs" / "toy_mira"
    if toy_root.exists():
        shutil.rmtree(toy_root)
    g = torch.Generator().manual_seed(SEED)
    yy, xx = torch.meshgrid(torch.linspace(-1, 1, H_TOY),
                            torch.linspace(-1, 1, H_TOY), indexing="ij")
    for t in range(T_TOY):
        cx, cy = (torch.rand(2, generator=g) * 1.2 - 0.6).tolist()   # blob centre
        w = 0.15 + 0.25 * torch.rand(1, generator=g).item()          # blob width
        m = torch.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * w * w)))  # (H,H) in [0,1]
        m = (2 * m - 1).float()                                            # -> [-1,1]
        # truth and draws are exchangeable draws of N(m_t, TAU^2) -> calibrated
        src  = (m + TAU * torch.randn(H_TOY, H_TOY, generator=g)).clamp(-1, 1)
        post = (m[None, None] +
                TAU * torch.randn(S_TOY, 1, H_TOY, H_TOY, generator=g)).clamp(-1, 1)
        d = {"post": post, "src": src[None], "obs": src[None].clone(),
             "pick": t, "src_name": f"toy_{t:02d}", "noise_sigma": TAU,
             "steps": 0, "image_pool": 2}
        run = toy_root / f"pick{t:02d}" / "samples"
        run.mkdir(parents=True, exist_ok=True)
        torch.save(d, run / "posterior_draws.pt")
    _run_dirs = discover_runs()
    MODELS = {"toy": _run_dirs}
    HAVE_DATA = len(_run_dirs) > 0
    print(f"Wrote {T_TOY} toy runs under {toy_root}")
    print(f"discovered runs = {len(_run_dirs)}, HAVE_DATA = {HAVE_DATA}")
else:
    print("MAKE_TOY_DATA is False -> using whatever was discovered in Section 2.")

## 5. Load posterior draws

Each `samples/posterior_draws.pt` holds one true source (`src`) and its draws
(`post`, shape `(S, 1, H, W)`). We key truths by `src_name` so every model is
scored on the SAME sources, flatten images to `(H*W,)`, and trim to a common
number of samples `S`.

In [ ]:
def _load_run(run_dir):
    d = torch.load(Path(run_dir) / "samples" / "posterior_draws.pt",
                   map_location="cpu", weights_only=False)
    post = d["post"].float().squeeze(1)                 # (S, H, W)
    src  = d["src"].float().squeeze()                   # (H, W)
    name = d.get("src_name", Path(run_dir).name)
    return name, src.reshape(-1), post.reshape(post.shape[0], -1)

if HAVE_DATA:
    per_model, truth_by_name = {}, {}
    for mname, dirs in MODELS.items():
        per_model[mname] = {}
        for rd in dirs:
            name, src_flat, post_flat = _load_run(rd)
            per_model[mname][name] = post_flat
            truth_by_name.setdefault(name, src_flat)

    # Truths common to every model (MIRA needs a shared truth set).
    common = sorted(set.intersection(*[set(m) for m in per_model.values()]))
    if not common:
        HAVE_DATA = False
        print("No sources are shared across all models -> nothing to score.")
    else:
        S = min(per_model[m][n].shape[0] for m in per_model for n in common)
        D = truth_by_name[common[0]].numel()
        model_names = list(per_model.keys())
        M, T = len(model_names), len(common)
        print(f"models M={M}, truths T={T}, samples/truth S={S}, pixel dim D={D}")
        if T < 5:
            print("WARNING: very few truths -- MIRA is only meaningful with many "
                  "source picks. Sample more --pick values for a real test.")
        truth_px = torch.stack([truth_by_name[n] for n in common], dim=0)   # (T, D)
        post_px  = torch.stack([torch.stack([per_model[m][n][:S] for n in common])
                                for m in model_names], dim=0)               # (M, T, S, D)
else:
    print("No data -> skipping real-data load.")

## 6. Build the MIRA feature space

Distance-based scores degrade in very high dimensions, so (like the PQMass PCA
cross-check) we project both truths and posterior samples onto the top principal
components of the **combined** stack -- a neutral basis -- to get `q = PCA_DIMS`.

In [ ]:
if HAVE_DATA:
    if USE_PCA:
        k = min(PCA_DIMS, D)
        stack = torch.cat([truth_px, post_px.reshape(-1, D)], dim=0).numpy()
        mean  = stack.mean(0, keepdims=True)
        _, _, Vt = np.linalg.svd(stack - mean, full_matrices=False)
        comps  = torch.from_numpy(Vt[:k]).float()               # (k, D)
        mean_t = torch.from_numpy(mean).float()
        proj = lambda x: (x - mean_t) @ comps.T
        truth_q = proj(truth_px)                                # (T, k)
        post_q  = proj(post_px.reshape(-1, D)).reshape(M, T, S, k)
        print(f"MIRA space: PCA({k})")
    else:
        truth_q, post_q = truth_px, post_px
        print(f"MIRA space: raw pixels ({D}) -- high-dimensional, interpret with care")

## 7. Run MIRA (+ bootstrap)

Optionally append a **calibration ruler**: the first model's posterior deliberately
shrunk (over-confident) and inflated (under-confident) about its per-truth mean. A
reasonably calibrated real posterior should score at or above these.

In [ ]:
if HAVE_DATA:
    names, post_all = list(model_names), post_q
    if ADD_BASELINES:
        base = post_q[0:1]                              # (1, T, S, q)
        mu   = base.mean(dim=2, keepdim=True)
        overc  = mu + 0.4 * (base - mu)
        underc = mu + 2.5 * (base - mu)
        post_all = torch.cat([post_q, overc, underc], dim=0)
        names = names + [f"{model_names[0]} [over-conf 0.4x]",
                         f"{model_names[0]} [under-conf 2.5x]"]

    score, std = mira(truth_q, post_all, num_runs=NUM_RUNS, norm=NORM, device=device)
    bmean, bstd = mira_bootstrap(truth_q, post_all, num_bootstrap=NUM_BOOTSTRAP,
                                 num_runs=1, norm=NORM, device=device)
    print(f"{'model':34s} {'MIRA':>8s} {'std':>8s} {'boot_std':>9s}")
    for n, m, sd, bs in zip(names, score.tolist(), std.tolist(), bstd.tolist()):
        print(f"{n:34s} {m:8.4f} {sd:8.4f} {bs:9.4f}")

## 8. Plot the scores

In [ ]:
if HAVE_DATA:
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(1.7 * len(names) + 3, 4))
    ax.bar(x, score.cpu().numpy(), yerr=bstd.cpu().numpy(), capsize=4, color="steelblue")
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha="right")
    ax.set_ylabel("MIRA score")
    ax.set_title(f"MIRA  (T={T}, S={S}, " + (f"PCA{min(PCA_DIMS, D)}" if USE_PCA else f"{D}px") + ")")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

## 9. Interpretation

- **Higher MIRA = better-calibrated posterior** (confirmed by the smoke test in
  Section 3). Compare a model against the over-/under-confident baselines: a
  well-calibrated posterior should sit at or above them, and clearly above both
  extremes.
- **Model comparison**: with several entries in `MODELS` (same sources), the
  model with the highest MIRA has the most accurate conditional distribution.
- **Caveats**
  - MIRA needs **many truths** (source picks). A handful of picks gives a noisy,
    under-powered score -- widen the sampling before drawing conclusions.
  - Scoring runs in **PCA space** by default; a raw-pixel cross-check
    (`USE_PCA = False`) should tell the same story. Disagreement usually means
    the pixel-space test is under-powered.
  - For high-dimensional image spaces you can also pass prior draws as the MIRA
    reference geometry via `center_choice` / `reference_choice` (see
    `mira_score.mira`) instead of the default random centers -- e.g. load
    `outputs/.../prior_check/prior_samples.pt` and project it with the same PCA
    basis.